# 02 · Hypnodensity analysis

This notebook uses only the tables persisted by `01_run_batch.ipynb`. It does not open MAT files or rerun staging and feature extraction.

## i) Imports

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from IPython.display import display

from dmt_hypnodensities import (
    add_hypnodensity_entropy, assemble_outputs, fit_mixed_models, load_config,
    paired_condition_wilcoxon, pairwise_stager_correlations,
    plot_condition_change_violins, plot_entropy_distribution,
    plot_hypnodensity_condition_violins, plot_paired_condition_changes,
    plot_stager_correlation_heatmap, prepare_epoch_cohen_d, prepare_run,
    prepare_within_condition_changes, save_figure, save_table,
    summarize_hypnodensities,
)

## ii) Parameters and persisted run

In [ ]:
CONFIG_PATH = next(path.resolve() for path in (Path('configs/analysis.yaml'), Path('../configs/analysis.yaml')) if path.is_file())
RUN_NAME = 'gap_sensitivity_090s_gssc_yasa_sleepfm_cpu_v1'
STAGES = ('W', 'N1', 'N2', 'N3', 'R')
PROBABILITIES = tuple(f'prob_{stage}' for stage in STAGES)
STAGERS = ('gssc', 'yasa', 'sleepfm')

run = prepare_run(load_config(CONFIG_PATH), RUN_NAME, reuse_existing=True)
tables = assemble_outputs(run.recordings, strict=True)
hypnodensities = add_hypnodensity_entropy(tables.hypnodensities)
display(hypnodensities.shape, hypnodensities.groupby('stager').size())

## iii) Analytical table

The preserved analysis unit is recording–block–epoch–electrode–stager. `before`, `after`, and `late` remain epoch labels rather than block boundaries.

In [ ]:
save_table(hypnodensities, run.tables / 'hypnodensities_with_entropy.parquet')
hypnodensity_summary = summarize_hypnodensities(tables.hypnodensities)
save_table(hypnodensity_summary, run.tables / 'hypnodensity_summary.csv')
display(hypnodensity_summary.head())

## iv) Statistics

In [ ]:
wilcoxon = paired_condition_wilcoxon(
    hypnodensities, value_columns=(*PROBABILITIES, 'entropy')
)
changes = prepare_within_condition_changes(
    hypnodensities,
    value_columns=(*PROBABILITIES, 'entropy'),
    delta_types=('abs', 'rel'),
)
cohen_d = prepare_epoch_cohen_d(hypnodensities, value_columns=PROBABILITIES)
mixed_models = fit_mixed_models(
    hypnodensities,
    outcomes=(*PROBABILITIES, 'entropy'),
    fixed_effects='C(condition) * C(experimental_label)',
    variance_components={'electrode': '0 + C(electrode)'},
    stratify_by=('stager',),
)
stager_correlations = pairwise_stager_correlations(
    tables.hypnodensities, stagers=STAGERS, include_entropy=True
)

for name, table in {
    'wilcoxon': wilcoxon,
    'hypnodensity_changes': changes,
    'hypnodensity_cohen_d': cohen_d,
    'hypnodensity_mixed_models': mixed_models,
    'stager_correlations': stager_correlations,
}.items():
    save_table(table, run.tables / f'{name}.csv')
display(wilcoxon, mixed_models, stager_correlations)

## v) Figuras

In [ ]:
figure, _ = plot_stager_correlation_heatmap(stager_correlations)
save_figure(figure, run.figures / 'stager_correlations')
plt.show()

figure, _ = plot_entropy_distribution(hypnodensities)
save_figure(figure, run.figures / 'entropy_by_stager')
plt.show()

In [ ]:
for stager in STAGERS:
    figure, _ = plot_hypnodensity_condition_violins(
        hypnodensities, wilcoxon_results=wilcoxon, stager=stager
    )
    save_figure(figure, run.figures / f'hypnodensity_conditions_{stager}')
    plt.show()

    figure, _ = plot_condition_change_violins(cohen_d, stager=stager)
    save_figure(figure, run.figures / f'hypnodensity_cohen_d_{stager}')
    plt.show()

    figure, _ = plot_paired_condition_changes(changes, stager=stager)
    save_figure(figure, run.figures / f'hypnodensity_paired_changes_{stager}')
    plt.show()